# TweetAge v7: cortes ordinales sobre modelos KN

Cinco cortes separan bajo (edad <= corte) y alto (edad > corte). Cada lado aprende sus propios modelos de lenguaje de caracteres y palabras, desde cero. Se reutilizan la normalizacion, el modelo ModifiedKN y la particion agrupada de v6, sin cambiar los archivos anteriores.

Lecciones: v5 tenia textos normalizados compartidos entre desarrollo y holdout; v6 los separo por grupos. El ajuste de sesgos/umbrales sobre OOF produce un indicador optimista y no equivale a una evaluacion independiente. Los datos suministrados son la unica fuente de entrenamiento; se excluyen vectorizadores, representaciones externas, modelos preentrenados y sklearn.linear_model.

Caracteres orden 4, palabras orden 3, lambda=0.75, sin division por longitud. No se impone coherencia monotona a los cinco scores reales: se informa su incumplimiento. La prediccion pedida es el numero de cortes cuyo score supera estrictamente su umbral, siempre entre 0 y 5.

Solo se abre el holdout y se genera CSV si el F1 OOF calibrado supera 0.44353 en al menos 0.002 usando exactamente los mismos grupos/pliegues que v6. Comparar dos indicadores calibrados no elimina su optimismo ni el efecto de seleccion. No se envia nada a Kaggle.

In [ ]:
import ast
import csv
import gc
import hashlib
import html
import json
import math
import platform
import re
import time
from collections import Counter
from functools import lru_cache
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from IPython.display import display, Markdown

SEED = 42
np.random.seed(SEED)
CLASSES = np.array(['13-17', '18-24', '25-34', '35-49', '50-64', '65-xx'])
CHAR_ORDER, WORD_ORDER, WEIGHT = 4, 3, 0.75
BASE_EPSILON = 1e-6
BOS, EOS, UNK = '\x02', '\x03', '\x01'
V6_OOF_BEFORE, V6_OOF_AFTER, MIN_GAIN = 0.44106, 0.44353, 0.002
roots = [Path.cwd(), Path.cwd() / 'workshops' / 'Kaggle-tweet-age', Path.cwd() / 'natural-language-processing-workshops' / 'workshops' / 'Kaggle-tweet-age']
OUT_DIR = next((candidate for candidate in roots if (candidate / 'TweetAge_v7_ordinal_kn.ipynb').is_file()), None)
assert OUT_DIR is not None, 'Ejecute desde la carpeta del notebook o la raiz del repo.'
NOTEBOOK_PATH = OUT_DIR / 'TweetAge_v7_ordinal_kn.ipynb'
SUB_PATH = OUT_DIR / 'age_submission_v7.csv'
SPLIT_PATH = OUT_DIR / 'split_idx_v6_seed42.npz'
DATA_DIR = OUT_DIR / 'tweet-age-2026-puj'
assert not SUB_PATH.exists(), 'La salida ya existe; no se sobrescribe.'
INPUT_HASHES = {name: hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest() for name in ['age_train.csv','age_test.csv','age_submission.csv']}
PROTECTED_HASHES = {name: hashlib.sha256((OUT_DIR / name).read_bytes()).hexdigest() for name in ['TweetAge_v5_charLM_kn.ipynb','TweetAge_v6_charLM_kn.ipynb','split_idx_v6_seed42.npz','requirements.txt']}
print('Python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__, '| sklearn', sklearn.__version__)
print('Semilla', SEED, '| Notebook', NOTEBOOK_PATH.name, '| CSV condicional', SUB_PATH.name)
print('SHA-256 age_train.csv:', INPUT_HASHES['age_train.csv'])
def check_compliance(notebook):
    forbidden = {'tfidfvectorizer', 'countvectorizer', 'hashingvectorizer', 'dictvectorizer', 'feature_extraction', 'linear_model', 'transformers', 'transformer', 'embedding', 'embeddings', 'word2vec', 'glove'}
    problems = []
    for cell_number, cell in enumerate(notebook['cells'], 1):
        if cell['cell_type'] != 'code':
            continue
        source = ''.join(cell['source']) if isinstance(cell['source'], list) else cell['source']
        for node in ast.walk(ast.parse(source)):
            names = []
            if isinstance(node, ast.Name):
                names = [node.id]
            elif isinstance(node, ast.Attribute):
                names = [node.attr]
            elif isinstance(node, ast.Import):
                names = [alias.name for alias in node.names]
            elif isinstance(node, ast.ImportFrom):
                names = [node.module or ''] + [alias.name for alias in node.names]
            for name in names:
                if any(part.lower() in forbidden for part in name.split('.')):
                    problems.append((cell_number, node.lineno, name))
            if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in {'eval', 'exec', '__import__'}:
                problems.append((cell_number, node.lineno, 'ejecucion dinamica'))
    if problems:
        raise RuntimeError(f'Guardia AST: codigo prohibido {problems}')
    return True

with NOTEBOOK_PATH.open(encoding='utf-8') as handle:
    assert check_compliance(json.load(handle))
print('Guardia AST de v6 superada para todas las celdas de codigo antes de entrenar.')


In [ ]:
def verify_csv(path, columns, delimiter, bom):
    with path.open('rb') as handle:
        prefix = handle.read(32768)
    first_line = prefix.decode('utf-8-sig').splitlines()[0]
    actual = csv.Sniffer().sniff(first_line, delimiters=',;\t|').delimiter
    header = next(csv.reader([first_line], delimiter=actual))
    assert actual == delimiter and prefix.startswith(b'\xef\xbb\xbf') == bom, (path.name, actual)
    assert header == columns, (path.name, header)
    print(path.name, 'BOM=', bom, 'separador=', repr(actual), 'columnas=', header)
    return header

URL_RE = re.compile(r'https?://[^\s]+|www\.[^\s]+', re.IGNORECASE)

MENTION_RE = re.compile(r'@\w+', re.UNICODE)

REPEAT_RE = re.compile(r'([^\W\d_])\1{3,}', re.UNICODE)

def replace_url(match):
    value = match.group(0)
    core = value.rstrip('.,!?;:)]}')
    return '<url>' + value[len(core):]

def normalize_text(text):
    return REPEAT_RE.sub(lambda match: match.group(1) * 3, MENTION_RE.sub('<user>', URL_RE.sub(replace_url, text)))

verify_csv(DATA_DIR / 'age_train.csv', ['id','text','age_range'], ';', True)
reference_columns = verify_csv(DATA_DIR / 'age_submission.csv', ['id','age_range'], ',', False)
train_raw = pd.read_csv(DATA_DIR / 'age_train.csv', sep=';', encoding='utf-8-sig', dtype=str, keep_default_na=False)
assert list(train_raw.columns) == ['id','text','age_range']
clean_text = train_raw.text.fillna('').str.strip().map(html.unescape)
excluded = clean_text.eq('') | clean_text.str.contains('#\u00bfNOMBRE', regex=False)
train = train_raw.loc[~excluded].copy()
train['raw_row'] = np.flatnonzero(~excluded.to_numpy())
train['text'] = clean_text.loc[~excluded]
train = train.reset_index(drop=True)
assert train.id.is_unique and set(train.age_range) == set(CLASSES)
train['normalized'] = train.text.map(normalize_text)
y_all = pd.Categorical(train.age_range, categories=CLASSES).codes.astype(np.int64)
with np.load(SPLIT_PATH, allow_pickle=False) as saved:
    split = {name: saved[name].copy() for name in saved.files}
assert str(split['train_sha256']) == INPUT_HASHES['age_train.csv'], 'SHA-256 del train distinto al NPZ; detener.'
assert int(split['seed']) == SEED
assert np.array_equal(split['retained_raw_idx'], train.raw_row.to_numpy())
idx_development, idx_holdout = split['train_idx'], split['valid_idx']
assert np.array_equal(np.sort(np.concatenate([idx_development,idx_holdout])), np.arange(len(train)))
assert not np.intersect1d(idx_development,idx_holdout).size
actual_groups, unique_texts = pd.factorize(train.normalized, sort=True)
assert np.array_equal(actual_groups,split['group_ids']), 'Normalizacion o grupos distintos de v6.'
assert np.array_equal(train.raw_row.to_numpy()[idx_development],split['raw_train_idx'])
assert np.array_equal(train.raw_row.to_numpy()[idx_holdout],split['raw_valid_idx'])
dev_groups, holdout_groups = split['train_groups'],split['valid_groups']
assert set(actual_groups[idx_development]) == set(dev_groups)
assert set(actual_groups[idx_holdout]) == set(holdout_groups)
intersection = set(train.normalized.iloc[idx_development]) & set(train.normalized.iloc[idx_holdout])
assert not intersection
texts_development = train.normalized.iloc[idx_development].to_numpy()
y_development = y_all[idx_development]
groups_development = actual_groups[idx_development]
group_counts = np.zeros((len(unique_texts),6),dtype=np.int64)
np.add.at(group_counts,(actual_groups,y_all),1)
group_targets = group_counts.argmax(axis=1)
print('NPZ reutilizado, sin regenerar:', SPLIT_PATH.name, '| SHA-256 correcto')
print('Train original/limpio/excluidos:', len(train_raw),len(train),int(excluded.sum()))
print('Desarrollo grupos/filas:',len(dev_groups),len(idx_development),'| Holdout grupos/filas:',len(holdout_groups),len(idx_holdout))
print('Interseccion de textos normalizados:',len(intersection),'| Solo URL train:',int(train.normalized.str.strip().eq('<url>').sum()))
display(train.age_range.value_counts().reindex(CLASSES).rename('filas').to_frame())


In [ ]:
TOKEN_RE = re.compile(r'<url>|<user>|\w+|[^\w\s]', re.UNICODE)

def tokenize(text, level):
    return list(text) if level == 'char' else TOKEN_RE.findall(text.lower())

def make_encoder(texts, level):
    frequencies = Counter()
    for text in texts:
        frequencies.update(tokenize(text, level))
    cutoff = 1 if level == 'char' else 2
    return {token: chr(index + 256) for index, token in enumerate(sorted(token for token, count in frequencies.items() if count >= cutoff))}

def encode_text(text, level, mapping):
    return ''.join(mapping.get(token, UNK) for token in tokenize(text, level))

def estimate_discounts(counts):
    histogram = Counter(counts.values())
    count1, count2, count3, count4 = [histogram.get(value, 0) for value in [1, 2, 3, 4]]
    factor = count1 / (count1 + 2 * count2) if count1 + 2 * count2 else 0.0
    first = 1 - 2 * factor * count2 / count1 if count1 and count2 else 0.75
    second = 2 - 3 * factor * count3 / count2 if count1 and count2 else 0.75
    third = 3 - 4 * factor * count4 / count3 if count1 and count2 and count3 else 0.75
    values = tuple(float(min(bucket - 1e-6, max(1e-6, value))) for bucket, value in enumerate([first, second, third], 1))
    return values, dict(N1=count1, N2=count2, N3=count3, N4=count4)

def discount_for(count, discounts):
    return discounts[min(count, 3) - 1] if count > 0 else 0.0

class ModifiedKN:
    def __init__(self, max_order, support):
        self.max_order = max_order
        self.support = set(support) | {EOS, UNK}

    def fit(self, sequences):
        highest = max(2, self.max_order)
        self.raw = {order: Counter() for order in range(2, highest + 1)}
        for sequence in sequences:
            padded = BOS * (highest - 1) + sequence + EOS
            for order, counts in self.raw.items():
                counts.update(padded[position - order + 1:position + 1] for position in range(highest - 1, len(padded)))
        self.continuation = {}
        for order in range(1, highest):
            effective = Counter()
            for gram in self.raw[order + 1]:
                effective[gram[1:]] += 1
            self.continuation[order] = effective
        self.base = self.continuation[1]
        self.base_total = sum(self.base.values()) + BASE_EPSILON * len(self.support)
        self.discount_records = []
        def prepare(count_tables, origin):
            discounts_by_order, histories_by_order = {}, {}
            for order, counts in count_tables.items():
                if order == 1:
                    continue
                discounts, histogram = estimate_discounts(counts)
                discounts_by_order[order] = discounts
                totals, removed = Counter(), Counter()
                for gram, count in counts.items():
                    totals[gram[:-1]] += count
                    removed[gram[:-1]] += discount_for(count, discounts)
                histories_by_order[order] = {history: (total, removed[history]) for history, total in totals.items()}
                self.discount_records.append(dict(origen=origin, orden=order, D1=discounts[0], D2=discounts[1], D3=discounts[2], **histogram))
            return discounts_by_order, histories_by_order
        self.raw_discounts, self.raw_hist = prepare(self.raw, 'frecuencias')
        self.cont_discounts, self.cont_hist = prepare(self.continuation, 'continuaciones')
        self.lower_cached = lru_cache(maxsize=200000)(self._lower)
        self.log_cached = lru_cache(maxsize=200000)(self._log_gram)
        return self

    def _lower(self, gram):
        if len(gram) == 1:
            return (self.base.get(gram, 0) + BASE_EPSILON) / self.base_total
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total, removed = self.cont_hist[order].get(gram[:-1], (0, 0.0))
        if not total:
            return lower
        count = self.continuation[order].get(gram, 0)
        return (count - discount_for(count, self.cont_discounts[order]) + removed * lower) / total

    def probability(self, gram):
        if len(gram) == 1:
            return self.lower_cached(gram)
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total, removed = self.raw_hist[order].get(gram[:-1], (0, 0.0))
        if not total:
            return lower
        count = self.raw[order].get(gram, 0)
        return (count - discount_for(count, self.raw_discounts[order]) + removed * lower) / total

    def _log_gram(self, gram):
        probability = self.probability(gram)
        if not 0 < probability <= 1 + 1e-10:
            raise ArithmeticError((repr(gram), probability))
        return math.log(probability)

    def log_probability(self, sequence, order):
        assert 1 <= order <= self.max_order
        padded = BOS * (order - 1) + sequence + EOS
        grams = Counter(padded[position - order + 1:position + 1] for position in range(order - 1, len(padded)))
        return sum(count * self.log_cached(gram) for gram, count in grams.items())

    def close(self):
        self.lower_cached.cache_clear()
        self.log_cached.cache_clear()

def test_modified_kn():
    values, _ = estimate_discounts(Counter({'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 1, 'f': 2, 'g': 3, 'h': 1}))
    assert len(set(values)) == 3 and all(0 < value < bucket for bucket, value in enumerate(values, 1))
    model = ModifiedKN(4, set('abc')).fit(['ab', 'ab', 'ac', 'bc', 'abc', 'abc', 'abc'])
    assert model.base['a'] == 1 and model.raw[2][BOS + 'a'] == 6
    for order in [1, 2, 3, 4]:
        contexts = set(model.raw_hist.get(order, {})) | {'', 'zz', BOS * (order - 1)}
        for context in contexts:
            history = (BOS * (order - 1) + context)[-(order - 1):] if order > 1 else ''
            assert abs(sum(model.probability(history + symbol) for symbol in model.support) - 1) < 1e-9, (order, repr(history))
        assert math.isfinite(model.log_probability('a' + UNK, order))
    assert model.log_probability('ab', 4) > model.log_probability('cc', 4)
    model.close()
    assert tokenize('Hola HOLA', 'word') == ['hola', 'hola']
    assert tokenize('Hola', 'char')[0] == 'H'
    print('Pruebas MKN: tres descuentos, normalizacion, continuaciones, BOS/EOS, UNK y capitalizacion correctos.')

In [ ]:
timing_rows = []

def cut_level_scores(fit_texts, fit_labels, app_texts, level, order, phase, fold):
    started = time.perf_counter()
    mapping = make_encoder(fit_texts,level)
    encoded_fit = [encode_text(text,level,mapping) for text in fit_texts]
    encoded_app = [encode_text(text,level,mapping) for text in app_texts]
    preprocessing = time.perf_counter()-started
    scores = np.zeros((len(app_texts),5),dtype=float)
    priors = np.zeros(5)
    for cut in range(5):
        cut_started = time.perf_counter()
        high_mask = fit_labels > cut
        high_count,low_count = int(high_mask.sum()),int((~high_mask).sum())
        assert high_count > 0 and low_count > 0
        priors[cut] = math.log(high_count/low_count)
        for high_side in [False,True]:
            sequences = (sequence for sequence,is_high in zip(encoded_fit,high_mask) if bool(is_high)==high_side)
            model = ModifiedKN(order,set(mapping.values())).fit(sequences)
            values = np.array([model.log_probability(sequence,order) for sequence in encoded_app])
            scores[:,cut] += values if high_side else -values
            model.close()
            del model
            gc.collect()
        elapsed = time.perf_counter()-cut_started+preprocessing/5
        timing_rows.append(dict(fase=phase,pliegue=fold,nivel=level,corte=cut,segundos=elapsed))
        print(phase,fold,level,'corte',cut,'segundos',round(elapsed,2),flush=True)
    return scores,priors

def ordinal_cut_scores(fit_texts,fit_labels,app_texts,phase,fold):
    characters,prior_char = cut_level_scores(fit_texts,fit_labels,app_texts,'char',CHAR_ORDER,phase,fold)
    words,prior_word = cut_level_scores(fit_texts,fit_labels,app_texts,'word',WORD_ORDER,phase,fold)
    assert np.array_equal(prior_char,prior_word)
    combined = WEIGHT*characters+(1-WEIGHT)*words+prior_char
    return characters,words,combined,prior_char

def decode(scores,thresholds):
    scores,thresholds = np.asarray(scores),np.asarray(thresholds)
    assert scores.ndim==2 and scores.shape[1]==5 and thresholds.shape==(5,)
    return np.sum(scores > thresholds,axis=1).astype(np.int64)

def macro_fast(targets,predictions):
    matrix = np.bincount(targets*6+predictions,minlength=36).reshape(6,6)
    denominator = matrix.sum(axis=0)+matrix.sum(axis=1)
    return float(np.divide(2*matrix.diagonal(),denominator,out=np.zeros(6),where=denominator!=0).mean())


In [ ]:
test_modified_kn()
synthetic_scores = np.array([[2.0 if cut < label else -2.0 for cut in range(5)] for label in range(6)])
assert np.array_equal(decode(synthetic_scores,np.zeros(5)),np.arange(6))
assert (np.diff(synthetic_scores,axis=1)<=0).all()
assert np.array_equal(decode(np.zeros((1,5)),np.zeros(5)),[0])
assert np.array_equal(decode(np.ones((1,5)),np.zeros(5)),[5])
assert np.all((decode(synthetic_scores,np.zeros(5))>=0)&(decode(synthetic_scores,np.zeros(5))<=5))
probe_counts = Counter({'a':1,'b':2,'c':3,'d':4,'e':1,'f':2,'g':3,'h':1})
probe_discounts,_ = estimate_discounts(probe_counts)
assert all(0<value<bucket for bucket,value in enumerate(probe_discounts,1))
probe_targets = np.arange(6)
assert macro_fast(probe_targets,probe_targets)==f1_score(probe_targets,probe_targets,average='macro')==1.0
print('Pruebas correctas: distribuciones normalizadas, descuentos validos, decoder 0..5 y caso sintetico monotono.')


In [ ]:
pilot_groups,_ = train_test_split(dev_groups,train_size=0.1,stratify=group_targets[dev_groups],random_state=SEED)
pilot_fit_groups,pilot_app_groups = train_test_split(pilot_groups,test_size=0.2,stratify=group_targets[pilot_groups],random_state=SEED)
pilot_fit = np.flatnonzero(np.isin(groups_development,pilot_fit_groups))
pilot_app = np.flatnonzero(np.isin(groups_development,pilot_app_groups))
assert not set(texts_development[pilot_fit]) & set(texts_development[pilot_app])
pilot_started = time.perf_counter()
pilot_char,pilot_word,pilot_combined,pilot_prior = ordinal_cut_scores(texts_development[pilot_fit],y_development[pilot_fit],texts_development[pilot_app],'piloto_10pct',0)
pilot_seconds = time.perf_counter()-pilot_started
projected_cv_seconds = pilot_seconds*(len(y_development)/len(pilot_fit))*0.8*5
SEARCH_FACTORS = [-2.0,-1.0,-0.5,0.0,0.5,1.0,2.0]
SEARCH_PASSES = [2.0,1.0,0.5,0.25]
reductions = []
if projected_cv_seconds > 1200:
    SEARCH_FACTORS = [-1.0,0.0,1.0]
    SEARCH_PASSES = [1.0,0.5]
    reductions.append('Proyeccion >1200 s: umbrales reducidos de 7 candidatos x 4 pasadas a 3 x 2. Ordenes y cinco cortes se mantienen fijos por el metodo; esta reduccion no abarata el entrenamiento de los KN.')
print('Piloto filas ajuste/validacion:',len(pilot_fit),len(pilot_app))
print('Piloto segundos:',round(pilot_seconds,2),'| Proyeccion CV cinco pliegues:',round(projected_cv_seconds,2),'s; estimacion lineal, no medicion completa.')
print('Reducciones:',reductions or 'ninguna')
display(pd.DataFrame(timing_rows))
del pilot_char,pilot_word,pilot_combined
gc.collect()


In [ ]:
cv_started = time.perf_counter()
group_folds = list(StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED).split(dev_groups,group_targets[dev_groups]))
folds = []
oof_char = np.zeros((len(y_development),5))
oof_word = np.zeros((len(y_development),5))
oof_combined = np.zeros((len(y_development),5))
oof_prior = np.zeros((len(y_development),5))
fold_id = np.full(len(y_development),-1,dtype=int)
for fold_number,(fit_group_positions,app_group_positions) in enumerate(group_folds,1):
    fit_idx = np.flatnonzero(np.isin(groups_development,dev_groups[fit_group_positions]))
    app_idx = np.flatnonzero(np.isin(groups_development,dev_groups[app_group_positions]))
    assert not set(texts_development[fit_idx]) & set(texts_development[app_idx])
    folds.append((fit_idx,app_idx))
    chars,words,combined,prior = ordinal_cut_scores(texts_development[fit_idx],y_development[fit_idx],texts_development[app_idx],'CV',fold_number)
    oof_char[app_idx],oof_word[app_idx],oof_combined[app_idx] = chars,words,combined
    oof_prior[app_idx] = prior
    fold_id[app_idx] = fold_number
    print('Pliegue',fold_number,'grupos disjuntos; filas validacion',len(app_idx),flush=True)
    del chars,words,combined
    gc.collect()
assert (fold_id>0).all()
assert np.isfinite(oof_char).all() and np.isfinite(oof_word).all()
assert np.allclose(oof_combined,WEIGHT*oof_char+(1-WEIGHT)*oof_word+oof_prior)
cv_seconds = time.perf_counter()-cv_started
print('CV segundos:',round(cv_seconds,2))
display(pd.DataFrame(timing_rows))


In [ ]:
def tune_thresholds(scores,targets):
    thresholds = np.zeros(5)
    before = macro_fast(targets,decode(scores,thresholds))
    best_f1 = before
    scales = np.clip(np.median(np.abs(scores),axis=0),0.01,100.0)
    history = []
    for pass_factor in SEARCH_PASSES:
        for cut in range(5):
            other_predictions = decode(scores,thresholds)-(scores[:,cut]>thresholds[cut])
            chosen = thresholds[cut]
            candidates = thresholds[cut]+pass_factor*scales[cut]*np.asarray(SEARCH_FACTORS)
            for candidate in candidates:
                predictions = other_predictions+(scores[:,cut]>candidate)
                result = macro_fast(targets,predictions.astype(np.int64))
                if result > best_f1+1e-12:
                    best_f1,chosen = result,float(candidate)
            thresholds[cut] = chosen
            history.append(dict(pasada=pass_factor,corte=cut,umbral=chosen,f1_macro=best_f1))
    return thresholds,before,macro_fast(targets,decode(scores,thresholds)),history

calibration_started = time.perf_counter()
best_thresholds,oof_before,oof_after,threshold_history = tune_thresholds(oof_combined,y_development)
oof_pred = decode(oof_combined,best_thresholds)
assert oof_after >= oof_before
assert abs(oof_after-f1_score(y_development,oof_pred,labels=np.arange(6),average='macro',zero_division=0))<1e-12
print('OOF F1 macro antes:',oof_before,'| despues:',oof_after)
print('El segundo es optimista: las etiquetas OOF se usan para calibrar umbrales; busqueda local, no optimo global garantizado.')
print('Busqueda por coordenadas: inicio cero, escalas por mediana absoluta de score, pasadas',SEARCH_PASSES,'candidatos relativos',SEARCH_FACTORS)
print('Umbrales congelados:',best_thresholds.tolist())
print('Calibracion segundos:',round(time.perf_counter()-calibration_started,2))
print('Filas con cortes binarios no monotonos:',int(np.any(np.diff((oof_combined>best_thresholds).astype(np.int8),axis=1)>0,axis=1).sum()))
display(pd.DataFrame(threshold_history))
display(pd.DataFrame({'pliegue':range(1,6),'f1_antes':[macro_fast(y_development[app_idx],decode(oof_combined[app_idx],np.zeros(5))) for _,app_idx in folds],'f1_despues':[macro_fast(y_development[app_idx],oof_pred[app_idx]) for _,app_idx in folds]}))

In [ ]:
gate_target = V6_OOF_AFTER+MIN_GAIN
PASA = bool(oof_after >= gate_target)
print('PASA' if PASA else 'NO PASA')
print('v7 OOF antes/despues:',round(oof_before,5),round(oof_after,5),'| v6:',V6_OOF_BEFORE,V6_OOF_AFTER)
print('Ganancia frente al mejor v6:',round(oof_after-V6_OOF_AFTER,5),'| Minima:',MIN_GAIN,'| Objetivo:',round(gate_target,5))
print('Mismo NPZ, normalizacion, grupos, estratos mayoritarios y cinco pliegues de v6; no se altera el holdout.')
if not PASA:
    print('NO PASA: se omiten explicitamente la evaluacion holdout y la prediccion final; no se creara age_submission_v7.csv.')


In [ ]:
if PASA:
    assert 'holdout_f1' not in globals(), 'Holdout ya evaluado; no reutilizar.'
    holdout_started = time.perf_counter()
    texts_holdout = train.normalized.iloc[idx_holdout].to_numpy()
    _,_,holdout_scores,_ = ordinal_cut_scores(texts_development,y_development,texts_holdout,'holdout',0)
    holdout_pred = decode(holdout_scores,best_thresholds)
    y_holdout = y_all[idx_holdout]
    holdout_f1 = f1_score(y_holdout,holdout_pred,labels=np.arange(6),average='macro',zero_division=0)
    print('F1 macro holdout:',holdout_f1)
    report = classification_report(y_holdout,holdout_pred,labels=np.arange(6),target_names=CLASSES,output_dict=True,zero_division=0)
    display(pd.DataFrame(report).T.loc[CLASSES,['precision','recall','f1-score','support']])
    display(pd.DataFrame(confusion_matrix(y_holdout,holdout_pred,labels=np.arange(6)),index=CLASSES,columns=CLASSES))
    print('Diferencias descriptivas v7-v5:',holdout_f1-0.44945,'v7-v6:',holdout_f1-0.41847,'; v5 usa otro split.')
    print('Holdout segundos:',round(time.perf_counter()-holdout_started,2))
else:
    print('HOLDOUT OMITIDO: NO PASA. No se entrenan modelos ni se calculan metricas de holdout.')


In [ ]:
if PASA:
    final_started = time.perf_counter()
    verify_csv(DATA_DIR / 'age_test.csv',['id','text'],';',True)
    test = pd.read_csv(DATA_DIR / 'age_test.csv',sep=';',encoding='utf-8-sig',dtype=str,keep_default_na=False)
    assert list(test.columns)==['id','text'] and test.id.is_unique and test.id.ne('').all()
    test_texts = test.text.fillna('').str.strip().map(html.unescape).map(normalize_text).to_numpy()
    _,_,test_scores,_ = ordinal_cut_scores(train.normalized.to_numpy(),y_all,test_texts,'final',0)
    submission = pd.DataFrame({'id':test.id,'age_range':CLASSES[decode(test_scores,best_thresholds)]})
    assert len(submission)==len(test)==112892
    assert list(submission.columns)==reference_columns==['id','age_range']
    assert submission.id.is_unique and submission.id.tolist()==test.id.tolist()
    assert submission.age_range.isin(CLASSES).all() and submission.age_range.ne('').all()
    with SUB_PATH.open('x',encoding='utf-8',newline='') as handle:
        submission.to_csv(handle,index=False,sep=',')
    with SUB_PATH.open('rb') as handle: prefix=handle.read(64)
    assert not prefix.startswith(b'\xef\xbb\xbf') and prefix.splitlines()[0]==b'id,age_range'
    print('CSV verificado:',SUB_PATH.name,'filas',len(submission),'IDs iguales y en orden, clases validas, coma, sin BOM.')
    print('Final segundos:',round(time.perf_counter()-final_started,2))
else:
    assert not SUB_PATH.exists()
    print('PREDICCION FINAL OMITIDA: NO PASA. age_test.csv no se carga para predecir y no se genera CSV.')


In [ ]:
if PASA:
    error_targets,error_predictions = y_holdout,holdout_pred
    error_indices,error_texts = idx_holdout,texts_holdout
    error_origin = 'holdout'
else:
    error_targets,error_predictions = y_development,oof_pred
    error_indices,error_texts = idx_development,texts_development
    error_origin = 'OOF desarrollo'

def interpret_error(text,real,predicted):
    if text.strip()=='<url>': return 'Hipotesis: una URL sin contenido lexico no aporta pistas suficientes de edad.'
    if len(TOKEN_RE.findall(text))<5: return 'Hipotesis: texto demasiado corto para distinguir cinco cortes.'
    if abs(real-predicted)==1: return 'Hipotesis: rangos contiguos comparten vocabulario y estilo.'
    if '<user>' in text: return 'Hipotesis: normalizar cuentas elimina pistas de comunidad.'
    return 'Hipotesis: el contenido es comun a edades distintas o los cortes no son coherentes.'

positions = np.flatnonzero(error_targets!=error_predictions)[:10]
errors = pd.DataFrame([dict(texto=train.text.iloc[error_indices[position]],real=CLASSES[error_targets[position]],predicha=CLASSES[error_predictions[position]],interpretacion=interpret_error(error_texts[position],int(error_targets[position]),int(error_predictions[position]))) for position in positions])
print('Diez errores de',error_origin,'; interpretaciones no causales.')
with pd.option_context('display.max_colwidth',180): display(errors)


## Referencias y estado de verificacion

- Peng, F., Schuurmans, D., Keselj, V. y Wang, S. (2003). **Language Independent Authorship Attribution with Character Level N-Grams**. EACL, abril de 2003. [ACL Anthology E03-1053](https://aclanthology.org/E03-1053/). Titulo, autores, ano y venue confirmados; la ficha no incluye paginas. **Verificar antes de publicar** la paginacion. El titulo solicitado con “using ... language models” no coincide con la ficha y no se conserva como titulo bibliografico exacto.

Google Scholar redirigio a bloqueo automatizado las otras seis consultas. Se retiraron de la bibliografia formal por no poder confirmar ano, venue y paginas en las fuentes requeridas durante esta ejecucion: Chen y Goodman (1999), Frank y Hall (2001), Baccianella et al. (2009), Teahan y Harper (2003), Schler et al. (2006) y Rao et al. (2010). Esta exclusion no afirma que los trabajos no existan. **Verificar antes de publicar** esas seis referencias y la paginacion pendiente de Peng. No se inventan ni completan paginas a partir de memoria.

El modelo se reutiliza de v6 sin cambios; la falta de verificacion bibliografica en linea no constituye una verificacion nueva de su metodologia.

In [ ]:
v7_holdout = holdout_f1 if PASA else np.nan
summary_table = pd.DataFrame({'version':['v5 historica','v6 agrupada','v7 ordinal'],'OOF_antes':[0.43539,V6_OOF_BEFORE,oof_before],'OOF_despues':[0.43847,V6_OOF_AFTER,oof_after],'holdout':[0.44945,0.41847,v7_holdout]})
display(summary_table)
summary_text = f"""## Resumen final

Regla: **{'PASA' if PASA else 'NO PASA'}**. OOF antes/despues: **{oof_before:.5f} / {oof_after:.5f}**; referencia calibrada v6 **{V6_OOF_AFTER:.5f}**, objetivo **{gate_target:.5f}**. Umbrales: `{best_thresholds.tolist()}`.

El holdout {'se evaluo una vez con umbrales congelados' if PASA else 'no se evaluo porque no paso la regla'}. {'Se creo age_submission_v7.csv.' if PASA else 'No se creo age_submission_v7.csv.'}

V5 tenia fuga por textos normalizados y usa un split distinto. V6 y v7 usan el mismo NPZ agrupado, los mismos grupos y pliegues; los indicadores OOF tras calibracion son optimistas. Los cortes se entrenan por separado y pueden no ser monotonos. La diferencia de scores entre metodologias no demuestra causalidad. El score publico de Kaggle no es comparable directamente con el holdout.

Entorno .venv-v6 dentro del proyecto. No se modificaron datos, modelos anteriores, requirements ni NPZ; no se regenero la particion. No se hicieron commit, push ni envios a Kaggle. Referencias: una parcialmente confirmada en ACL y seis retiradas por bloqueo de Scholar, pendientes de verificar antes de publicar.
"""
display(Markdown(summary_text))
assert INPUT_HASHES == {name:hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest() for name in INPUT_HASHES}
assert PROTECTED_HASHES == {name:hashlib.sha256((OUT_DIR/name).read_bytes()).hexdigest() for name in PROTECTED_HASHES}
print('Hashes de entradas y archivos protegidos: SIN CAMBIOS.')
